In [6]:
# Inisialisasi & Persiapan Data

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Tugas5").getOrCreate()

# Membaca data dari HDFS dan menghitung pendapatan
hdfs_path = "hdfs://localhost:9000/user/zerouno/tugas5/transaksi_tugas5.csv" 
df_transaksi = spark.read.csv(hdfs_path, header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# Membuat df_target dari dictionary
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"]
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

In [7]:
# A. Join & Perbandingam
# Aggregasi total pendapatan per kota
df_pendapatan_kota = df_transaksi.groupBy("kota").agg(_sum("pendapatan").alias("total_pendapatan"))

# Join dengan df_target dan hitung pencapaian_persen
df_bagian_a = df_pendapatan_kota.join(df_target, on="kota", how="inner") \
    .withColumn("pencapaian_persen", (col("total_pendapatan") / col("target_bulanan")) * 100) \
    .orderBy(col("pencapaian_persen").desc())

df_bagian_a.show()

[Stage 7:>                                                          (0 + 4) / 4]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [8]:
# B. Window Function — Kategori Terlaris per Kota

# Menghitung total pendapatan per kota dan kategori
df_kategori_kota = df_transaksi.groupBy("kota", "kategori") \
    .agg(_sum("pendapatan").alias("total_pendapatan"))

# Spesifikasi window per kota diurutkan berdasarkan pendapatan tertinggi
window_spec = Window.partitionBy("kota").orderBy(col("total_pendapatan").desc())

# Pemeringkatan top-1 dengan row_number()
df_bagian_b = df_kategori_kota.withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn") \
    .orderBy("kota")

df_bagian_b.show()

+----------+--------------------+----------------+
|      kota|            kategori|total_pendapatan|
+----------+--------------------+----------------+
|  Magelang|Kesehatan & Kecan...|         7275000|
| Purworejo|Kesehatan & Kecan...|        10075000|
|  Semarang|        Rumah Tangga|        11125000|
|      Solo|Kesehatan & Kecan...|         8425000|
|Yogyakarta|             Fashion|        13325000|
+----------+--------------------+----------------+



In [9]:
# C. SparkSQL

# Mendaftarkan temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Kueri SQL murni untuk menghitung jumlah transaksi per kota
df_bagian_c = spark.sql("""
    SELECT 
        t.kota, 
        tg.pic_cabang, 
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

df_bagian_c.show()

[Stage 15:===========================================>              (3 + 1) / 4]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



*D. Kesimpulan*

Berdasarkan hasil analisis data transaksi dan perbandingan terhadap target bulanan pada Bagian A dan B, cabang dengan kinerja terbaik adalah Purworejo yang dipimpin oleh Fitri. Cabang ini berhasil melampaui target secara signifikan dengan pencapaian tertinggi sebesar 152.17%. Total pendapatan yang berhasil dibukukan mencapai Rp 45.650.000 dari target awal sebesar Rp 30.000.000. Penjualan di cabang Purworejo didorong secara dominan oleh kategori Kesehatan & Kecantikan yang menyumbang pendapatan terbesar sebesar Rp 10.075.000.

Sebaliknya, cabang yang paling memerlukan perhatian khusus dari pihak manajemen adalah Semarang di bawah arahan Sari. Cabang ini mencatatkan performa terendah dengan pencapaian target hanya sebesar 69.41%. Total pendapatan yang diperoleh baru mencapai Rp 38.175.000 dari target bulanan sebesar Rp 55.000.000. Meskipun kategori Rumah Tangga menjadi penyumbang terbesar di kota Semarang dengan nilai Rp 11.125.000, angka tersebut belum cukup untuk mengejar deviasi target yang cukup besar. Manajemen disarankan untuk melakukan evaluasi strategi pemasaran di Semarang agar dapat mendongkrak penjualan kategori lain secara lebih optimal.